In [1]:
import pandas as pd
import numpy as np
df = pd.read_excel('/content/cases_train2 (11).xlsx')
df['Оценка'] = df['Оценка'].round().astype('Int64')
df_test_cases = pd.read_excel('/content/Test1_2 (3).xlsx')
df_test_cases['Оценка'] = df_test_cases['Оценка'].round().astype('Int64')
#df = pd.read_excel("/content/good table.xlsx")
df.head(100)


,Кейс,Решение,Решение кейса,ЦА,Проработка решения,Финансовая модель и метрики,Анализ рисков,Доказательства,Оценка
0,1,1,Я выбрал отрасль туризма и гостиничного бизнес...,1,1,1,1,1,1
1,1,2,Я выбрал для разработки отраслевого решения сф...,2,2,2,2,2,2
2,1,4,"Я выбрал отрасль образования, а именно сегмент...",2,4,4,3,3,3
3,1,5,Я выбрал отрасль розничной торговли продуктами...,1,3,2,2,1,2
4,1,6,Я выбрал для отраслевого решения сферу гостини...,3,2,2,2,3,2
...,...,...,...,...,...,...,...,...,...
95,1,97,Я выбрал отрасль грузоперевозок. Целевая аудит...,2,2,4,3,1,2
96,1,98,Я выбрал отрасль ремонта техники. Целевая ауди...,3,2,3,2,2,2
97,1,99,Я выбрал отрасль такси. Целевая аудитория: вод...,1,1,4,3,1,2
98,1,100,Я выбрал отрасль ремонта квартир. Целевая ауди...,3,3,3,2,2,3


In [2]:
import torch
import torch.nn as nn
import numpy as np
from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')


In [3]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    mean_absolute_error,
    confusion_matrix
)

import warnings
warnings.filterwarnings('ignore')

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:
df["Кейс"].value_counts()

,count
Кейс,
1,454
8,128
2,121
4,120
6,120
5,120
10,120
9,120
3,118


In [ ]:
df["Доказательства"].value_counts()

,count
Доказательства,
4,394
2,323
3,320
5,289
1,206
6,1


In [ ]:
df["Оценка"].value_counts()

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error


In [6]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [7]:
class BertClassifier(nn.Module):
  def __init__(self, num_classes, model_name='bert-base-multilingual-cased'):
    super(BertClassifier, self).__init__()
    self.bert = BertModel.from_pretrained(model_name)
    for param in self.bert.parameters():
      param.requires_grad=False
    self.classifier = nn.Sequential(nn.Dropout(0.3),
                                    nn.Linear(self.bert.config.hidden_size, 256),
                                    nn.ReLU(),
                                    nn.Dropout(0.2),
                                    nn.Linear(256, num_classes)
    )
  def forward(self, input_ids, attention_mask):
    with torch.no_grad():
      outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
      cls = outputs.last_hidden_state[:, 0, :]
    logits = self.classifier(cls)
    return logits

In [8]:
tokenizer= BertTokenizer.from_pretrained("bert-base-multilingual-cased")
model = BertClassifier(num_classes=5)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=2e-4)
criterion = nn.CrossEntropyLoss()


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
def train(loader):
    total_loss = 0
    model.train()
    for batch in loader:
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = batch["labels"].to(device)
      optimizer.zero_grad()
      outputs = model(input_ids=input_ids, attention_mask=attention_mask)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
    return total_loss / len(loader)


In [10]:
def evals(loader):
    model.eval()
    loss_lst = []
    predictions = []
    total = 0
    correct = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            preds = torch.argmax(outputs, dim=1)

            loss_lst.append(loss.item())
            correct += (preds == labels).sum().item()
            total += len(labels)
            predictions.extend(preds.cpu().numpy())

    return np.mean(loss_lst), correct / total, np.array(predictions)

In [11]:
def predicts(texts, model, tokenizer, device, label_encoder=None, max_len=128):
  model.eval()
  predictions = []
  with torch.no_grad():
    for text in texts:
      encoding = tokenizer(text, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
      input_ids = encoding["input_ids"].to(device)
      attention_mask = encoding["attention_mask"].to(device)
      outputs = model(input_ids=input_ids, attention_mask=attention_mask)
      probs = torch.softmax(outputs, dim=1)
      pred = torch.argmax(probs, dim=1)
      predictions.append(pred.cpu().numpy()[0])
  return np.array(predictions)

In [12]:
X_text_audience = df['Решение кейса'].fillna('').values
y_audience = (df['ЦА'].values - 1).astype("int64")

X_train_text_audience, X_test_text_audience, y_train_audience, y_test_audience = train_test_split(
    X_text_audience, y_audience, test_size=0.2, random_state=42, stratify=y_audience
)
X_text_dataset_audience = TextDataset(X_train_text_audience, y_train_audience, tokenizer)
X_val_dataset_audience = TextDataset(X_test_text_audience, y_test_audience, tokenizer)
X_text_dataloader_audience = DataLoader(X_text_dataset_audience, batch_size=16, shuffle=True)
X_val_dataloader_audience = DataLoader(X_val_dataset_audience, batch_size=16, shuffle=False)
model_audience = BertClassifier(num_classes=5)
model_audience.to(device)
optimizer_audience = torch.optim.AdamW(model_audience.classifier.parameters(), lr=2e-4)
criterion_audience = nn.CrossEntropyLoss()
model = model_audience
model.to(device)
optimizer = optimizer_audience
criterion = criterion_audience


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
best_loss_audience = float("inf")
loss_all = 0
loss_val = 0
for epoch in range(10):
  train_loss = train(X_text_dataloader_audience)
  loss_all += train_loss
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_audience)
  loss_val+=val_loss
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  if val_loss <= best_loss_audience:
    best_loss_audience = val_loss
    torch.save(model_audience.state_dict(), "best_model_audience.pt")
print("best_val_loss", best_loss_audience)
print("avg_train", loss_all / 10)
print("avg_val", loss_val / 10)

train_loss= 1.5656247324757762
val_loss= 1.529564243555069
val_acc= 0.2736156351791531
train_loss= 1.5279297797710865
val_loss= 1.497015917301178
val_acc= 0.28338762214983715
train_loss= 1.4959604802069726
val_loss= 1.4762473404407501
val_acc= 0.3127035830618892
train_loss= 1.4659989158828537
val_loss= 1.4574012219905854
val_acc= 0.32247557003257327
train_loss= 1.451561029855307
val_loss= 1.4378008186817168
val_acc= 0.3355048859934853
train_loss= 1.4235355652771986
val_loss= 1.4209631145000459
val_acc= 0.33876221498371334
train_loss= 1.4069915777677064
val_loss= 1.4088761329650878
val_acc= 0.31596091205211724
train_loss= 1.4005290904602448
val_loss= 1.4062179625034332
val_acc= 0.34201954397394135
train_loss= 1.3807516175431092
val_loss= 1.4006187081336976
val_acc= 0.3322475570032573
train_loss= 1.3831604270191935
val_loss= 1.3703089773654937
val_acc= 0.3583061889250814
best_val_loss 1.3703089773654937
avg_train 1.450204321625945
avg_val 1.4405014437437056


In [14]:
model_audience.load_state_dict(torch.load("best_model_audience.pt"))
model_audience.to(device)
model_audience.eval()
_, _, predictions = evals(X_val_dataloader_audience)
true_labels = []
for batch in X_val_dataloader_audience:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.60      0.21      0.32        28
           1       0.29      0.13      0.18        55
           2       0.29      0.54      0.38        70
           3       0.36      0.42      0.39        83
           4       0.51      0.34      0.41        71

    accuracy                           0.36       307
   macro avg       0.41      0.33      0.33       307
weighted avg       0.39      0.36      0.35       307

MAE 0.8241042345276873


In [15]:
_, _, predictions_audience = evals(X_text_dataloader_audience)
true_labels_audience = []
for batch in X_text_dataloader_audience:
  labels = batch["labels"].cpu().numpy()
  true_labels_audience.extend(labels)
print(classification_report(true_labels_audience, predictions_audience))
print("MAE", mean_absolute_error(true_labels_audience, predictions_audience))

              precision    recall  f1-score   support

           0       0.12      0.06      0.08       114
           1       0.15      0.08      0.10       222
           2       0.21      0.38      0.27       279
           3       0.25      0.26      0.26       330
           4       0.22      0.16      0.18       281

    accuracy                           0.21      1226
   macro avg       0.19      0.19      0.18      1226
weighted avg       0.20      0.21      0.20      1226

MAE 1.3287112561174552


In [16]:
X_text_sol = df['Решение кейса'].fillna('').values
y_sol = (df['Проработка решения'].values - 1).astype("int64")

X_train_text_sol, X_test_text_sol, y_train_sol, y_test_sol = train_test_split(
    X_text_sol, y_sol, test_size=0.2, random_state=42, stratify=y_sol
)
X_text_dataset_sol = TextDataset(X_train_text_sol, y_train_sol, tokenizer)
X_val_dataset_sol = TextDataset(X_test_text_sol, y_test_sol, tokenizer)
X_text_dataloader_sol = DataLoader(X_text_dataset_sol, batch_size=16, shuffle=True)
X_val_dataloader_sol = DataLoader(X_val_dataset_sol, batch_size=16, shuffle=False)

In [17]:
model_sol = BertClassifier(num_classes=5)
model_sol.to(device)
optimizer_sol = torch.optim.AdamW(model_sol.classifier.parameters(), lr=2e-4)
criterion_sol = nn.CrossEntropyLoss()
model = model_sol
model.to(device)
optimizer = optimizer_sol
criterion = criterion_sol


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
y_sol.value_counts()

AttributeError: 'numpy.ndarray' object has no attribute 'value_counts'

In [19]:
best_loss_sol = float("inf")
train_all = 0
val_all = 0
for epoch in range(10):
  train_loss = train(X_text_dataloader_sol)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_sol)
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  train_all += train_loss
  val_all+= val_loss
  if val_loss <= best_loss_sol:
    best_loss_sol = val_loss
    torch.save(model_sol.state_dict(), "best_model_sol.pt")
print("avg_train=,", train_all/10)
print("avg_val=,", val_all/10)



train_loss= 1.5692972300888657
val_loss= 1.5370150327682495
val_acc= 0.254071661237785
train_loss= 1.5350528320708832
val_loss= 1.511173152923584
val_acc= 0.2671009771986971
train_loss= 1.5150242594929484
val_loss= 1.4818212270736695
val_acc= 0.3257328990228013
train_loss= 1.4847131394720696
val_loss= 1.4627894699573516
val_acc= 0.31921824104234525
train_loss= 1.4636830184366796
val_loss= 1.4630655229091645
val_acc= 0.3289902280130293
train_loss= 1.4377610559587355
val_loss= 1.4407830357551574
val_acc= 0.3289902280130293
train_loss= 1.4192252577125253
val_loss= 1.4181579172611236
val_acc= 0.36156351791530944
train_loss= 1.4143813058927461
val_loss= 1.4022052824497222
val_acc= 0.3485342019543974
train_loss= 1.4042250057319543
val_loss= 1.4044833838939668
val_acc= 0.34201954397394135
train_loss= 1.392536853815054
val_loss= 1.3922794699668883
val_acc= 0.3583061889250814
avg_train=, 1.463589995867246
avg_val=, 1.4513773494958877


In [21]:
model_sol.load_state_dict(torch.load("best_model_sol.pt"))
model_sol.to(device)
model_sol.eval()
_, _, predictions = evals(X_val_dataloader_sol)
true_labels = []
for batch in X_val_dataloader_sol:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.83      0.36      0.50        28
           1       0.36      0.48      0.41        62
           2       0.43      0.08      0.14        73
           3       0.32      0.81      0.46        77
           4       0.67      0.03      0.06        67

    accuracy                           0.36       307
   macro avg       0.52      0.35      0.31       307
weighted avg       0.48      0.36      0.29       307

MAE 0.8631921824104235


In [23]:
_, _, predictions_train = evals(X_text_dataloader_sol)
true_labels_train = []
for batch in X_text_dataloader_sol:
  labels = batch["labels"].cpu().numpy()
  true_labels_train.extend(labels)
print(classification_report(true_labels_train, predictions_train))
print("MAE", mean_absolute_error(true_labels_train, predictions_train))

              precision    recall  f1-score   support

           0       0.10      0.05      0.07       113
           1       0.21      0.31      0.25       248
           2       0.36      0.07      0.12       291
           3       0.26      0.63      0.37       309
           4       0.00      0.00      0.00       265

    accuracy                           0.24      1226
   macro avg       0.19      0.21      0.16      1226
weighted avg       0.20      0.24      0.18      1226

MAE 1.2707993474714518


In [24]:
model_finance = BertClassifier(num_classes=5)
model_finance.to(device)
optimizer_finance = torch.optim.AdamW(model_finance.classifier.parameters(), lr=2e-4)
criterion_finance = nn.CrossEntropyLoss()
model = model_finance
model.to(device)
optimizer = optimizer_finance
criterion = criterion_finance


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
X_text_finance = df['Решение кейса'].fillna('').values
y_finance = (df['Финансовая модель и метрики'].values - 1).astype("int64")

X_train_text_finance, X_test_text_finance, y_train_finance, y_test_finance = train_test_split(
    X_text_finance, y_finance, test_size=0.2, random_state=42,stratify= y_finance
)
X_text_dataset_finance = TextDataset(X_train_text_finance, y_train_finance, tokenizer)
X_val_dataset_finance = TextDataset(X_test_text_finance, y_test_finance, tokenizer)
X_text_dataloader_finance = DataLoader(X_text_dataset_finance, batch_size=16, shuffle=True)
X_val_dataloader_finance = DataLoader(X_val_dataset_finance, batch_size=16, shuffle=False)

In [26]:
best_loss_finance = float("inf")
all_train = 0
all_val= 0
for epoch in range(10):
  train_loss = train(X_text_dataloader_finance)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_finance)
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  all_train += train_loss
  all_val += val_loss
  if val_loss <= best_loss_finance:
    best_loss_finance = val_loss
    torch.save(model_finance.state_dict(), "best_model_finance.pt")
print("avg_val", all_val / 10)
print("avg_train", all_train/10)

train_loss= 1.5580949334355143
val_loss= 1.5326431274414063
val_acc= 0.247557003257329
train_loss= 1.5341468523075055
val_loss= 1.507191115617752
val_acc= 0.31921824104234525
train_loss= 1.5141278087318717
val_loss= 1.474018543958664
val_acc= 0.34527687296416937
train_loss= 1.4903449733535965
val_loss= 1.459494763612747
val_acc= 0.33876221498371334
train_loss= 1.4680091545179292
val_loss= 1.4386582016944884
val_acc= 0.3257328990228013
train_loss= 1.448222844631641
val_loss= 1.425203686952591
val_acc= 0.32247557003257327
train_loss= 1.4399625295168395
val_loss= 1.4235861480236054
val_acc= 0.3485342019543974
train_loss= 1.4312457233280331
val_loss= 1.4067099869251252
val_acc= 0.3745928338762215
train_loss= 1.4203837072694456
val_loss= 1.4189912796020507
val_acc= 0.3550488599348534
train_loss= 1.4165155268334724
val_loss= 1.3985189259052277
val_acc= 0.3811074918566775
avg_val 1.4485015779733659
avg_train 1.4721054053925848


In [27]:
model_finance.load_state_dict(torch.load("best_model_finance.pt"))
model_finance.to(device)
model_finance.eval()
_, _, predictions = evals(X_val_dataloader_finance)
true_labels = []
for batch in X_val_dataloader_finance:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.92      0.33      0.49        33
           1       0.36      0.39      0.38        69
           2       0.39      0.28      0.32        76
           3       0.35      0.72      0.47        80
           4       0.00      0.00      0.00        49

    accuracy                           0.38       307
   macro avg       0.40      0.35      0.33       307
weighted avg       0.37      0.38      0.34       307

MAE 0.8859934853420195


In [28]:
_, _, predictions_train = evals(X_text_dataloader_finance)
true_labels_train = []
for batch in X_text_dataloader_finance:
  labels = batch["labels"].cpu().numpy()
  true_labels_train.extend(labels)
print(classification_report(true_labels_train, predictions_train))
print("MAE", mean_absolute_error(true_labels_train, predictions_train))

              precision    recall  f1-score   support

           0       0.09      0.04      0.05       129
           1       0.22      0.23      0.22       276
           2       0.21      0.12      0.15       305
           3       0.25      0.55      0.34       321
           4       0.00      0.00      0.00       195

    accuracy                           0.23      1226
   macro avg       0.15      0.19      0.15      1226
weighted avg       0.18      0.23      0.18      1226

MAE 1.2471451876019577


In [29]:
model_risks = BertClassifier(num_classes=5)
model_risks.to(device)
optimizer_risks = torch.optim.AdamW(model_risks.classifier.parameters(), lr=2e-4)
criterion_risks = nn.CrossEntropyLoss()
model = model_risks
model.to(device)
optimizer = optimizer_risks
criterion = criterion_risks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [30]:
X_text_risks = df['Решение кейса'].fillna('').values
y_risks = (df['Анализ рисков'].values - 1).astype("int64")

X_train_text_risks, X_test_text_risks, y_train_risks, y_test_risks = train_test_split(
    X_text_risks, y_risks, test_size=0.2, random_state=42, stratify=y_risks
)
X_text_dataset_risks = TextDataset(X_train_text_risks, y_train_risks, tokenizer)
X_val_dataset_risks = TextDataset(X_test_text_risks, y_test_risks, tokenizer)
X_text_dataloader_risks = DataLoader(X_text_dataset_risks, batch_size=16, shuffle=True)
X_val_dataloader_risks = DataLoader(X_val_dataset_risks, batch_size=16, shuffle=False)

In [31]:
best_loss_risks = float("inf")
all_val = 0
all_train = 0
for epoch in range(10):
  train_loss = train(X_text_dataloader_risks)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_risks)
  all_val += val_loss
  all_train += train_loss
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  if val_loss <= best_loss_risks:
    best_loss_risks = val_loss
    torch.save(model_risks.state_dict(), "best_model_risks.pt")
print("avg_val=", all_val / 10)
print("avg_train=", all_train/10)

train_loss= 1.5732604265213013
val_loss= 1.5368038952350616
val_acc= 0.30618892508143325
train_loss= 1.5513447167037369
val_loss= 1.5150170564651488
val_acc= 0.32247557003257327
train_loss= 1.521101072237089
val_loss= 1.4843746364116668
val_acc= 0.3289902280130293
train_loss= 1.494138900335733
val_loss= 1.4599241077899934
val_acc= 0.36156351791530944
train_loss= 1.4708721962842075
val_loss= 1.4364885151386262
val_acc= 0.34527687296416937
train_loss= 1.4584064050154253
val_loss= 1.4277433037757874
val_acc= 0.3583061889250814
train_loss= 1.4449253206129198
val_loss= 1.4327875256538392
val_acc= 0.38436482084690554
train_loss= 1.4381703850510832
val_loss= 1.406897872686386
val_acc= 0.3517915309446254
train_loss= 1.4186305473377179
val_loss= 1.3941146373748778
val_acc= 0.36482084690553745
train_loss= 1.4116871387927563
val_loss= 1.3942983329296113
val_acc= 0.36482084690553745
avg_val= 1.4488449883460999
avg_train= 1.4782537108891969


In [32]:
model_risks.load_state_dict(torch.load("best_model_risks.pt"))
model_risks.to(device)
model_risks.eval()
_, _, predictions = evals(X_val_dataloader_risks)
true_labels = []
for batch in X_val_dataloader_risks:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.81      0.32      0.46        41
           1       0.31      0.17      0.22        64
           2       0.31      0.50      0.38        80
           3       0.38      0.65      0.48        74
           4       0.00      0.00      0.00        48

    accuracy                           0.36       307
   macro avg       0.36      0.33      0.31       307
weighted avg       0.34      0.36      0.32       307

MAE 0.8501628664495114


In [33]:
_, _, predictions_train = evals(X_text_dataloader_risks)
true_labels_train = []
for batch in X_text_dataloader_risks:
  labels = batch["labels"].cpu().numpy()
  true_labels_train.extend(labels)
print(classification_report(true_labels_train, predictions_train))
print("MAE", mean_absolute_error(true_labels_train, predictions_train))

              precision    recall  f1-score   support

           0       0.11      0.06      0.08       166
           1       0.12      0.06      0.08       255
           2       0.26      0.46      0.33       319
           3       0.25      0.38      0.30       297
           4       0.00      0.00      0.00       189

    accuracy                           0.23      1226
   macro avg       0.15      0.19      0.16      1226
weighted avg       0.17      0.23      0.19      1226

MAE 1.2210440456769984


In [34]:
model_proves = BertClassifier(num_classes=5)
model_proves.to(device)
optimizer_proves = torch.optim.AdamW(model_proves.classifier.parameters(), lr=2e-4)
criterion_proves = nn.CrossEntropyLoss()
model = model_proves
model.to(device)
optimizer = optimizer_proves
criterion = criterion_proves


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
X_text_proves = df['Решение кейса'].fillna('').values
y_proves = (df['Доказательства'].values - 1).astype("int64")

X_train_text_proves, X_test_text_proves, y_train_proves, y_test_proves = train_test_split(
    X_text_proves, y_proves, test_size=0.2, random_state=42, stratify=y_proves
)
X_text_dataset_proves = TextDataset(X_train_text_proves, y_train_proves, tokenizer)
X_val_dataset_proves = TextDataset(X_test_text_proves, y_test_proves, tokenizer)
X_text_dataloader_proves = DataLoader(X_text_dataset_proves, batch_size=16, shuffle=True)
X_val_dataloader_proves = DataLoader(X_val_dataset_proves, batch_size=16, shuffle=False)

In [36]:
best_loss_proves = float("inf")
all_val = 0
all_train = 0
for epoch in range(10):
  train_loss = train(X_text_dataloader_proves)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_proves)
  all_val += val_loss
  all_train += train_loss
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  if val_loss <= best_loss_proves:
    best_loss_proves = val_loss
    torch.save(model_proves.state_dict(), "best_model_proves.pt")
print("avg_val=", all_val/10)
print("avg_train", all_train/10)

train_loss= 1.588138400734245
val_loss= 1.555145663022995
val_acc= 0.26058631921824105
train_loss= 1.5584717091027793
val_loss= 1.5300480425357819
val_acc= 0.30944625407166126
train_loss= 1.5183250811192897
val_loss= 1.4962318658828735
val_acc= 0.30293159609120524
train_loss= 1.4940767179835925
val_loss= 1.5020900011062621
val_acc= 0.2736156351791531
train_loss= 1.4730734345200773
val_loss= 1.4647180438041687
val_acc= 0.3257328990228013
train_loss= 1.4517953148135891
val_loss= 1.4507339715957641
val_acc= 0.33876221498371334
train_loss= 1.4409661525255675
val_loss= 1.4482745587825776
val_acc= 0.3355048859934853
train_loss= 1.4210492759555966
val_loss= 1.4322942793369293
val_acc= 0.34201954397394135
train_loss= 1.4001336484760434
val_loss= 1.4430227100849151
val_acc= 0.34527687296416937
train_loss= 1.3977450333632433
val_loss= 1.4329087853431701
val_acc= 0.31921824104234525
avg_val= 1.4755467921495435
avg_train 1.474377476859402


In [37]:
model_proves.load_state_dict(torch.load("best_model_proves.pt"))
model_proves.to(device)
model_proves.eval()
_, _, predictions = evals(X_val_dataloader_proves)
true_labels = []
for batch in X_val_dataloader_proves:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.81      0.32      0.46        41
           1       0.35      0.46      0.40        65
           2       0.00      0.00      0.00        64
           3       0.31      0.78      0.44        79
           4       0.00      0.00      0.00        58

    accuracy                           0.34       307
   macro avg       0.29      0.31      0.26       307
weighted avg       0.26      0.34      0.26       307

MAE 0.9739413680781759


In [38]:
_, _, predictions_train = evals(X_text_dataloader_proves)
true_labels_train = []
for batch in X_text_dataloader_proves:
  labels = batch["labels"].cpu().numpy()
  true_labels_train.extend(labels)
print(classification_report(true_labels_train, predictions_train))
print("MAE", mean_absolute_error(true_labels_train, predictions_train))

              precision    recall  f1-score   support

           0       0.11      0.07      0.08       165
           1       0.25      0.37      0.30       258
           2       0.50      0.00      0.01       257
           3       0.26      0.60      0.36       315
           4       0.25      0.00      0.01       231

    accuracy                           0.24      1226
   macro avg       0.27      0.21      0.15      1226
weighted avg       0.29      0.24      0.17      1226

MAE 1.360522022838499


In [ ]:
new_results_df = pd.DataFrame(columns=['Id','Текст_решения','Анализ ЦА','Проработка решения','Финансовая модель и метрики','Анализ рынков','Доказательства','Предсказанная_оценка','Дата'])

In [ ]:
import re

In [ ]:
dec = '''МОЙ ПРОЕКТ

Я хочу сделать бота для школьников. Он будет помогать решать задачи.

Целевая аудитория - школьники. Им это нужно для учебы.

Мое решение - бот в телеграме. Он будет бесплатный. Похожих ботов нет.

Финансы: разработка стоит примерно 500 тысяч рублей. Потом будем зарабатывать на рекламе.

Риски: могут появиться конкуренты. Будем делать лучше.

Доказательства: я сам учился в школе и знаю, что это нужно. Многие мои друзья тоже так думают.'''

In [39]:
def get_prediction_audience(text):
  model_audience.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  preds = predicts([text], model_audience, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [40]:
def get_prediction_solution(text):
  model_sol.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  preds = predicts([text], model_sol, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [41]:
def get_prediction_finance(text):
  model_finance.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  model.load_state_dict(torch.load("best_model_audience.pt", map_location=device))
  preds = predicts([text], model_finance, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [42]:
def get_prediction_risks(text):
  model_risks.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  preds = predicts([text], model_risks, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [43]:
def get_prediction_proves(text):
  model_proves.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  preds = predicts([text], model_proves, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [44]:
df_test_cases['pred_audience'] = df_test_cases['Решение кейса'].apply(get_prediction_audience)
df_test_cases['pred_sol'] = df_test_cases['Решение кейса'].apply(get_prediction_solution)
df_test_cases['pred_finance'] = df_test_cases['Решение кейса'].apply(get_prediction_finance)
df_test_cases['pred_risks'] = df_test_cases['Решение кейса'].apply(get_prediction_risks)
df_test_cases['pred_proves'] = df_test_cases['Решение кейса'].apply(get_prediction_proves)

In [45]:
df_test_cases['pred_audience']+=1
df_test_cases['pred_finance'] += 1
df_test_cases['pred_risks'] += 1
df_test_cases['pred_sol']+= 1
df_test_cases['pred_proves'] += 1

In [46]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)


In [47]:
#ЦА
accuracy_audience = accuracy_score(df_test_cases['ЦА'], df_test_cases['pred_audience'])
f1micro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='micro')
f1macro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='macro')
print('accuracy_audience= ', accuracy_audience)
print('f1micro_audience= ', f1micro_audience)
print('f1macro_audience= ', f1macro_audience)
print(classification_report(df_test_cases['ЦА'], df_test_cases['pred_audience']))
print("MAE=", mean_absolute_error(df_test_cases['ЦА'], df_test_cases['pred_audience']))

accuracy_audience=  0.3384615384615385
f1micro_audience=  0.3384615384615385
f1macro_audience=  0.29720310881601203
              precision    recall  f1-score   support

           1       0.43      0.13      0.20        23
           2       0.26      0.19      0.22        26
           3       0.30      0.74      0.43        34
           4       0.47      0.32      0.38        22
           5       0.67      0.16      0.26        25

    accuracy                           0.34       130
   macro avg       0.43      0.31      0.30       130
weighted avg       0.41      0.34      0.31       130

MAE= 0.8769230769230769


In [48]:
accuracy_sol = accuracy_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'])
f1micro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='micro')
f1macro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='macro')
print('accuracy_sol= ', accuracy_sol)
print('f1micro_sol= ', f1micro_sol)
print('f1macro_sol= ', f1macro_sol)
print(classification_report(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))
print("MAE=", mean_absolute_error(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))

accuracy_sol=  0.36923076923076925
f1micro_sol=  0.36923076923076925
f1macro_sol=  0.2939105829088851
              precision    recall  f1-score   support

           1       0.60      0.29      0.39        21
           2       0.33      0.66      0.44        32
           3       0.50      0.10      0.17        30
           4       0.35      0.72      0.47        25
           5       0.00      0.00      0.00        22

    accuracy                           0.37       130
   macro avg       0.36      0.35      0.29       130
weighted avg       0.36      0.37      0.30       130

MAE= 0.8538461538461538


In [49]:
accuracy_finance = accuracy_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'])
f1micro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='micro')
f1macro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='macro')
print('accuracy_finance= ', accuracy_finance)
print('f1micro_finance= ', f1micro_finance)
print('f1macro_finance= ', f1macro_finance)
print(classification_report(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))
print("MAE=", mean_absolute_error(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))

accuracy_finance=  0.35384615384615387
f1micro_finance=  0.35384615384615387
f1macro_finance=  0.2928595627279993
              precision    recall  f1-score   support

           1       0.83      0.22      0.34        23
           2       0.32      0.55      0.40        33
           3       0.26      0.19      0.22        31
           4       0.38      0.71      0.49        24
           5       0.00      0.00      0.00        19

    accuracy                           0.35       130
   macro avg       0.36      0.33      0.29       130
weighted avg       0.36      0.35      0.31       130

MAE= 0.8923076923076924


In [50]:
accuracy_risks = accuracy_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'])
f1micro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='micro')
f1macro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='macro')
print('accuracy_risks= ', accuracy_risks)
print('f1micro_risks= ', f1micro_risks)
print('f1macro_risks= ', f1macro_risks)
print(classification_report(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))
print("MAE=", mean_absolute_error(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))

accuracy_risks=  0.3
f1micro_risks=  0.3
f1macro_risks=  0.23178479912777342
              precision    recall  f1-score   support

           1       0.33      0.19      0.24        26
           2       0.35      0.22      0.27        32
           3       0.27      0.73      0.39        30
           4       0.42      0.19      0.26        27
           5       0.00      0.00      0.00        15

    accuracy                           0.30       130
   macro avg       0.27      0.27      0.23       130
weighted avg       0.30      0.30      0.26       130

MAE= 0.9153846153846154


In [51]:
accuracy_proves = accuracy_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'])
f1micro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='micro')
f1macro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='macro')
print('accuracy_proves= ', accuracy_proves)
print('f1micro_proves= ', f1micro_proves)
print('f1macro_proves= ', f1macro_proves)
print(classification_report(df_test_cases['Доказательства'], df_test_cases['pred_proves']))
print("MAE", mean_absolute_error(df_test_cases['Доказательства'], df_test_cases['pred_proves']))


accuracy_proves=  0.3230769230769231
f1micro_proves=  0.3230769230769231
f1macro_proves=  0.3037202155716088
              precision    recall  f1-score   support

           1       0.57      0.22      0.32        18
           2       0.58      0.29      0.39        38
           3       0.23      0.76      0.35        25
           4       0.33      0.26      0.29        19
           5       0.50      0.10      0.17        30

    accuracy                           0.32       130
   macro avg       0.44      0.33      0.30       130
weighted avg       0.46      0.32      0.31       130

MAE 0.9153846153846154


In [ ]:
sec_dec = """АНАЛИЗ ЦА:
Выделено 3 сегмента:
- Школьники 10-11 класс (45%)
- Студенты (35%)
- Учителя (20%)
Проведен опрос 100 человек.

РЕШЕНИЕ:
Экосистема из 2 продуктов: телеграм бот + дашборд для учителей.
Партнеры: Яндекс.Образование, МФТИ.

ФИНАНСЫ:
CAPEX: 2 млн руб. CAC = 100 руб, LTV = 2000 руб.
Окупаемость: 10 месяцев.

РИСКИ:
Технические (30%) - резервный сервер.

ДОКАЗАТЕЛЬСТВА:
Пилот в школе: NPS = 65.
"""

In [52]:
df_test_cases["Predict_score"] = round((df_test_cases['pred_audience'] + df_test_cases['pred_sol'] + df_test_cases['pred_finance'] + df_test_cases['pred_risks'] + df_test_cases['pred_proves'])/5)

In [53]:
print("MAE= ",mean_absolute_error(df_test_cases["Оценка"], df_test_cases["Predict_score"]))

MAE=  0.7384615384615385


In [ ]:
sol_test = """ Я выбрала отрасль кофеен и кофе-точек с собой (кофе на вынос, кофейные киоски, небольшие кофейни на 2–5 столиков). ЦА разделена на две группы. Первая группа — это микробизнес: кофе-байки и кофе-островки с одним сотрудником, работают как ИП или самозанятые. Их проблема в том, что кофейные зерна нужно покупать свежей обжарки каждую неделю, а хорошие обжарщики требуют предоплату 100% за партию от 5 кг, это около 20–30 тысяч рублей единовременно, что для маленькой точки с ежедневной выручкой 5–7 тысяч рублей чувствительно. Вторая группа — это малый бизнес: кофейни с посадочными местами и штатом 2–5 бариста. Их проблема в том, что сезонность спроса сильно различается: зимой продажи выше за счет горячих напитков, а летом люди покупают холодный кофе и чаще берут с собой, но в межсезонье (апрель и октябрь) падение выручки достигает 30%, при этом аренду и зарплату платить надо. Я опиралась на данные исследования «Рынок кофеен России 2024» от компании CoffeeData: рост рынка на 15% за год, количество кофеен достигло 12 тысяч, из них 70% — малый и микробизнес. Также я посмотрел открытую статистику по кофейному рынку на сайте Росстата и несколько статей в профильных телеграм-каналах. В качестве решения я предлагаю продукт «Альфа.Кофе»: кредит на закупку зёрен с отсрочкой первого платежа на 30 дней и с льготным периодом 0% на первые две недели, а также сезонный овердрафт на покрытие аренды в межсезонье на сумму до 100 тысяч рублей. Партнеры — два крупных обжарщика зерна «Coffe Lab» и «Torrefacto», с которыми можно договориться о более выгодных ценах для клиентов банка. Отличие от конкурентов в том, что ни у Сбера, ни у Т-Банка нет специального продукта для кофеен с отсрочкой именно под зерно. По финансам: разработка кредитного продукта обойдется примерно в 6 миллионов рублей, интеграция с партнерами-обжарщиками — в 2 миллиона, маркетинг в кофейных чатах и через конференции бариста — в 2 миллиона. Прогноз: 350 клиентов в первый год, средний доход с клиента — 20 тысяч рублей (за счет процентов по кредиту и эквайринга), выручка — 7 миллионов рублей. Окупаемость — примерно 2,5 года. Риски: конкурентный — другие банки могут запустить похожие продукты; кредитный — часть клиентов может не вернуть деньги, но мы будем проверять по выписке с кофемашины, кто сколько продает. В доказательство я использовал данные CoffeeData и результаты пары интервью с владельцами кофеен в своем городе."""

In [ ]:
predicted_score_audience = get_prediction_audience(sec_dec)
predicted_score_sol = get_prediction_solution(sec_dec)
predicted_score_finance = get_prediction_finance(sec_dec)
predicted_score_risks = get_prediction_risks(sec_dec)
predicted_score_proves = get_prediction_proves(sec_dec)
predicted_score_audience2 = get_prediction_audience(sec_dec)
predicted_score_sol2 = get_prediction_solution(sec_dec)
predicted_score_finance2 = get_prediction_finance(sec_dec)
predicted_score_risks2 = get_prediction_risks(sec_dec)
predicted_score_proves2 = get_prediction_proves(sec_dec)
new_row = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience,
    'Проработка решения': predicted_score_sol,
    'Финансовая модель и метрики': predicted_score_finance,
    'Анализ рынков': predicted_score_risks,
    'Доказательства': predicted_score_proves,
    'Предсказанная_оценка': round((predicted_score_audience + predicted_score_sol +
                      predicted_score_finance + predicted_score_risks +
                      predicted_score_proves) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_row2 = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience2,
    'Проработка решения': predicted_score_sol2,
    'Финансовая модель и метрики': predicted_score_finance2,
    'Анализ рынков': predicted_score_risks2,
    'Доказательства': predicted_score_proves2,
    'Предсказанная_оценка': round((predicted_score_audience2 + predicted_score_sol2 +
                      predicted_score_finance2 + predicted_score_risks2 +
                      predicted_score_proves2) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_results_df = pd.concat([new_results_df, new_row, new_row2], ignore_index=True)
new_results_df.to_excel("новая_таблица.xlsx", index=False)


/tmp/ipykernel_3876/1005490930.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_3876/1097642392.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)


In [ ]:
new_results_df

,Id,Текст_решения,Анализ ЦА,Проработка решения,Финансовая модель и метрики,Анализ рынков,Доказательства,Предсказанная_оценка,Дата
0,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
1,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
